# 2.5D unstructured LES solver on an A100: channel Re_τ 395

The unstructured collocated finite-volume solver in the x-y plane, Fourier in the periodic span, RK3 with a projection per stage, WALE, the whole step on CuPy (`src/upiso25.py`, `device="gpu"`). This notebook: (1) checks the environment, (2) profiles one step on this GPU and prints the run-time table, (3) runs the correctness check, (4) launches the channel Re_τ 395 case, (5) compares it with the Moser–Kim–Mansour 1999 DNS.

Run it from the repository root (the directory that holds `src/`, `meshes/`, `reference/`). The first code cell installs what is missing into this kernel (pyamg, cupy-cuda12x, ...); `tools/a100/requirements.txt` and `tools/a100/Dockerfile` do the same outside a notebook. The shell cells (`!python ...`) use the same interpreter as the kernel, so installs made here are visible to them.

In [ ]:
# 0. Dependencies into THIS kernel (pip of the running interpreter, not the shell's). pyamg builds the
#    multigrid hierarchy on the host; cupy-cuda12x is the GPU array library (binary wheels on x86-64).
import sys, subprocess, importlib
def ensure(mod, pkg):
    try: importlib.import_module(mod); print(f"ok  {mod}")
    except ImportError:
        print(f"installing {pkg} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg]); importlib.invalidate_caches(); importlib.import_module(mod); print(f"ok  {mod} (installed)")
ensure("numpy", "numpy>=1.26"); ensure("scipy", "scipy>=1.11"); ensure("pyamg", "pyamg>=5.0"); ensure("matplotlib", "matplotlib>=3.7")
try:
    import cupy; print("ok  cupy", cupy.__version__)
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"]); import cupy; print("ok  cupy (installed)", cupy.__version__)


In [ ]:
import os, sys, subprocess, time; sys.path.insert(0, os.getcwd())
import numpy as np, cupy as cp
p = cp.cuda.runtime.getDeviceProperties(0); print(p["name"], f"{p['totalGlobalMem']/2**30:.0f} GB, {p['multiProcessorCount']} SMs, cupy {cp.__version__}")
import pyamg, scipy; print("pyamg", pyamg.__version__, "scipy", scipy.__version__, "numpy", np.__version__)   # if this fails, rerun cell 0
for f in ("src/upiso25.py", "src/ucuda.py", "results/minchan_re180_field.npz", "results/fosls_chan180_stats_t5.2_30.npz", "reference/mkm_chan395/chan395.means"): print(("ok  " if os.path.exists(f) else "MISSING ") + f)

## 1. Profile one step on this GPU
The size ladder on a periodic box and the run-time table for the target cases at this device's measured rate. The step is memory-bandwidth bound past a ~50 ms launch floor (about 700 kernel launches per step from Python); on the GB10 the box rate was 77–112 ms per 10⁵ cell-modes and a real stretched mesh 256.

In [ ]:
!{sys.executable} tools/a100/profile_step.py --sizes 64x32,128x64,256x64,384x64

## 2. Correctness on this device (2 min)
The 3D Taylor–Green energy balance: −dE/dt must equal the scheme's own discrete dissipation to 0.1–0.2% at every sample (record section 51), E₀ = 31.006277.

In [ ]:
!TGV_DEVICE=gpu TGV_SOLVER=amg TGV_RE=100 TGV_T=2 TGV_N=32 TGV_NZ=32 TGV_DT=0.02 {sys.executable} test_utgv3d.py A

## 3. The channel Re_τ 395

Setup: δ = u_τ = 1, ν = 1/395, constant pressure gradient f_x = 1 (so u_τ = 1 by construction and the measured wall stress is a check), minimal box L_x = π, L_z = 0.34π (L_x⁺ = 1241, L_z⁺ = 422), 96 × 160 wall-clustered quads (Δx⁺ 12.9, Δy⁺ 1.0 at the wall) × 128 Fourier planes (Δz⁺ 3.3), dt 0.001, WALE. Initial condition: the Re_τ 180 DNS field (the same one the 180 case starts from) with its plane mean replaced by the MKM Re_τ 395 mean, so the flow starts turbulent at the right bulk velocity (U_b/u_τ = 17.54) and re-equilibrates over the first ~10 time units. Statistics from t = 10 to 30.

Cost: 30,000 steps. Measured on the GB10 at exactly this size: 2.1–2.8 s/step (18–23 h; 1.0M cell-modes at the real-mesh rate of ~230 ms per 10⁵, CFL 0.66). On an A100-80GB expect 0.35–0.75 s/step (3–6 h) from the bandwidth ratio; the profile cell above gives this device's own number. The run is launched in the background and logs to `results/logs/`; the cell below tails the log. Checkpoints every 5000 steps (`--restart results/<tag>_ckpt.npz` resumes).

Reference: MKM 1999 at Re_τ = 392.24 (full box 2π × π). A minimal box reproduces the near-wall statistics; the outer-region profiles (y⁺ > ~120) are box-dependent, so judge the log region and the peaks, not the centreline.

In [ ]:
tag = "uchan395_96x160x128_wale_cpg"
os.makedirs("results/logs", exist_ok=True)
cmd = f"nohup {sys.executable} -u run_uchannel25.py --device gpu --re-tau 395 --nx 96 --ny 160 --nz 128 --dt 0.001 --T 30 --t-stats 10 --report 1000 --checkpoint 5000 --tag {tag} > results/logs/{tag}.log 2>&1 &"
print(cmd); subprocess.Popen(cmd, shell=True); time.sleep(60); print(open(f"results/logs/{tag}.log").read()[-1500:])

In [ ]:
# progress: rerun this cell; each report line is 1 time unit
print(open(f"results/logs/{tag}.log").read()[-2500:])

## 4. Compare with MKM 1999
When the log shows `RESULT ...`. Mean profile, rms, Reynolds shear stress in wall units against the DNS; the criteria of the plan (V2): mean within 3% of u_τ in the log region, u_rms peak within 5%, Re_τ within 2%, pressure two-colour mode < 1%.

In [ ]:
import matplotlib.pyplot as plt
d = np.load(f"results/{tag}_stats.npz"); A = np.loadtxt("reference/mkm_chan395/chan395.means"); B = np.loadtxt("reference/mkm_chan395/chan395.reystress")
yd, Ud = A[:, 0], A[:, 2]; ypd = yd * 392.24; ud, vd, wd, uvd = np.sqrt(B[:, 2]), np.sqrt(B[:, 3]), np.sqrt(B[:, 4]), -B[:, 5]
yp, ut = d["yp"], float(d["ut"]); print(f"u_tau {ut:.4f}  Re_tau {float(d['re_tau']):.1f}  samples {int(d['nsamp'])}")
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
ax[0].semilogx(ypd[1:], Ud[1:], "k-", lw=2, label="MKM 1999 Re_tau 392"); ax[0].semilogx(yp, d["U"] / ut, "o-", ms=3, lw=1, label="2.5D LES"); yl = np.logspace(1, 2.5, 50); ax[0].semilogx(yl, np.log(yl) / 0.41 + 5.2, "0.5", ls="--", lw=0.8); ax[0].set(xlabel="y+", ylabel="U+"); ax[0].legend()
for arr, ls, nm in ((ud, "-", "u'"), (vd, "--", "v'"), (wd, ":", "w'")): ax[1].plot(ypd, arr, "k", ls=ls, lw=2, label=f"DNS {nm}")
for k, ls in (("urms", "-"), ("vrms", "--"), ("wrms", ":")): ax[1].plot(yp, d[k] / ut, "C0", ls=ls, lw=1.2)
ax[1].set(xlabel="y+", ylabel="rms / u_tau", xlim=(0, 395)); ax[1].legend()
ax[2].plot(ypd, uvd, "k-", lw=2, label="DNS"); ax[2].plot(yp, -d["uv"] / ut**2, "C0o-", ms=3, lw=1, label="2.5D LES"); ax[2].set(xlabel="y+", ylabel="-<u'v'>/u_tau^2", xlim=(0, 395)); ax[2].legend()
plt.tight_layout(); plt.savefig("figures/uchannel_re395_profiles.png", dpi=130); plt.show()
sel = (yp > 30) & (yp < 120); dU = d["U"][sel] / ut - np.interp(yp[sel], ypd, Ud)
print(f"log region 30<y+<120: U+ - DNS mean {dU.mean():+.3f} (max {np.abs(dU).max():.3f}) u_tau units;  u_rms+ peak {(d['urms']/ut).max():.3f} vs DNS {ud.max():.3f} ({((d['urms']/ut).max()/ud.max()-1)*100:+.1f}%);  -<uv>+ max {(-d['uv']/ut**2).max():.3f} vs {uvd.max():.3f}")

## 5. Notes
* `--forcing mf --Ub 17.54` runs at constant mass flow instead (Re_τ becomes an outcome).
* `--Lx 6.2832 --Lz 3.1416 --nx 192 --nz 256` is the MKM full box (4× the cost).
* Triangle meshes are not for this case: every triangle configuration failed the channel criteria (record section 55).
* The pressure two-colour indicator printed each report line is the mode that broke the structured code's channel; it must stay < 1% of p_rms.